In [0]:
%sql
-- Standard calendar (date dimension) table: 1940-01-01 to 2030-12-31
CREATE OR REPLACE TABLE caresync_databricks.silver.calender
COMMENT 'Standard calendar / date dimension from 1940-01-01 to 2030-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS d
)
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT)                         AS datekey,
  d                                                               AS date,
  year(d)                                                         AS year,
  CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END                       AS half_year,
  quarter(d)                                                      AS quarter,
  concat('Q', quarter(d))                                         AS quarter_name,
  concat(year(d), '-Q', quarter(d))                               AS year_quarter,
  month(d)                                                        AS month,
  date_format(d, 'MMMM')                                          AS month_name,
  date_format(d, 'MMM')                                           AS month_short_name,
  date_format(d, 'yyyy-MM')                                       AS year_month,
  CAST(date_format(d, 'yyyyMM') AS INT)                           AS year_month_key,
  weekofyear(d)                                                   AS week_of_year,
  extract(YEAROFWEEK FROM d)                                      AS iso_week_year,
  dayofmonth(d)                                                   AS day_of_month,
  dayofyear(d)                                                    AS day_of_year,
  weekday(d) + 1                                                  AS day_of_week,       -- 1 = Monday ... 7 = Sunday (ISO)
  date_format(d, 'EEEE')                                          AS day_name,
  date_format(d, 'EEE')                                           AS day_short_name,
  CASE WHEN weekday(d) >= 5 THEN true ELSE false END              AS is_weekend,
  CASE WHEN weekday(d) < 5 THEN true ELSE false END               AS is_weekday,
  date_sub(d, weekday(d))                                         AS week_start_date,   -- Monday
  date_add(date_sub(d, weekday(d)), 6)                            AS week_end_date,     -- Sunday
  trunc(d, 'MM')                                                  AS month_start_date,
  last_day(d)                                                     AS month_end_date,
  dayofmonth(last_day(d))                                         AS days_in_month,
  CAST(date_trunc('QUARTER', d) AS DATE)                          AS quarter_start_date,
  date_sub(add_months(CAST(date_trunc('QUARTER', d) AS DATE), 3), 1) AS quarter_end_date,
  trunc(d, 'YY')                                                  AS year_start_date,
  make_date(year(d), 12, 31)                                      AS year_end_date,
  CASE WHEN d = last_day(d) THEN true ELSE false END              AS is_month_end,
  CASE WHEN (year(d) % 4 = 0 AND year(d) % 100 <> 0) OR year(d) % 400 = 0
       THEN true ELSE false END                                   AS is_leap_year
FROM dates
ORDER BY d;

In [0]:
%sql
select * from caresync_databricks.silver.calender

In [0]:
%sql
-- Validate row count, date range and uniqueness
SELECT
  count(*)                 AS total_rows,
  count(DISTINCT datekey)  AS distinct_datekeys,
  min(date)                AS min_date,
  max(date)                AS max_date
FROM caresync_databricks.silver.calender;